In [1]:

from datetime import datetime
from pyspark.sql.types import *
import uuid

# === CONFIGURATION - Change for each notebook ===
NOTEBOOK_NAME = "ntk_silver_GroupMembership"        # ← Change this for each notebook
PIPELINE_NAME = "pipeline_test"     # ← Change this for each pipeline
ACTIVITY_TYPE = "DataTransformation"     # ← DataExtract/DataTransform/DataLoad/DataValidation
SOURCE_PATH = "abfss://Bronze/Group_Memberships" # ← Change source path
SOURCE_PATH = "abfss://Bronze/User_Group_Memberships" # ← Change source path
TARGET_PATH = "abfss://silver/Dim_Group_Memberships" # ← Change target path

def log_etl_activity(status, start_time=None, error=None, **metrics):
    """Log ETL activity to pipeline table"""
    current_time = datetime.now()
    
    # Get next LogID
    try:
        log_id = spark.sql("SELECT COALESCE(MAX(LogID), 0) + 1 as id FROM etl_silver_pipeline_log").collect()[0]['id']
    except:
        log_id = 1
    
    if status == "STARTED":
        data = [(
            log_id, PIPELINE_NAME, f"run_{current_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            current_time, None, None, "RUNNING", None, 
            SOURCE_PATH, TARGET_PATH, None, None, None, None, 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
        start_time = current_time
        
    else:
        duration = int((current_time - start_time).total_seconds()) if start_time else None
        data = [(
            log_id, PIPELINE_NAME, f"run_{start_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            start_time, current_time, duration, status, 
            str(error) if error else None, SOURCE_PATH, TARGET_PATH,
            metrics.get('rows_read'), metrics.get('rows_written'), 
            metrics.get('file_count'), metrics.get('bytes_processed'), 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
    
    # Schema for etl_silver_pipeline_log table
    schema = StructType([
        StructField("LogID", LongType()), StructField("PipelineName", StringType()),
        StructField("RunID", StringType()), StructField("ActivityName", StringType()),
        StructField("ActivityType", StringType()), StructField("NotebookName", StringType()),
        StructField("Sequence", IntegerType()), StructField("StartTime", TimestampType()),
        StructField("EndTime", TimestampType()), StructField("DurationSeconds", IntegerType()),
        StructField("Status", StringType()), StructField("ErrorMessage", StringType()),
        StructField("SourcePath", StringType()), StructField("TargetPath", StringType()),
        StructField("RowsRead", LongType()), StructField("RowsWritten", LongType()),
        StructField("FileCountProcessed", IntegerType()), StructField("BytesProcessed", LongType()),
        StructField("InsertedOn", TimestampType()), StructField("InsertedBy", StringType()),
        StructField("CorrelationID", StringType())
    ])
    
    # Save to table
    spark.createDataFrame(data, schema).write.mode("append").saveAsTable("etl_silver_pipeline_log")
    
    # Print status
    if status == "STARTED":
        print(f"🚀 Starting {NOTEBOOK_NAME}")
    elif status == "SUCCESS":
        duration_text = f" ({duration}s)" if duration else ""
        print(f"✅ {NOTEBOOK_NAME} completed successfully{duration_text}")
    else:
        print(f"❌ {NOTEBOOK_NAME} failed")
    
    return current_time if status == "STARTED" else None

# Start logging
print(f"🔧 Initializing {NOTEBOOK_NAME}...")
start_time = log_etl_activity("STARTED")

# Initialize variables for tracking metrics
rows_read = 0
rows_written = 0
file_count = 0
bytes_processed = 0

# print(f"📊 Ready to process data from: {SOURCE_PATH}")
# print(f"🎯 Target location: {TARGET_PATH}")

StatementMeta(, 6dc5032b-58cb-4cfe-a66c-cb1917b73da0, 3, Finished, Available, Finished)

🔧 Initializing ntk_silver_GroupMembership...
🚀 Starting ntk_silver_GroupMembership


In [2]:
source_path = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Bronze_lakehouse.Lakehouse/Files/Bronze_layer/SharePointFiles"
target_path = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting"

StatementMeta(, 6dc5032b-58cb-4cfe-a66c-cb1917b73da0, 4, Finished, Available, Finished)

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, lit, current_timestamp, to_timestamp, when, upper, regexp_replace, coalesce
)
from pyspark.sql.types import StringType, IntegerType
from datetime import datetime

# Initialize Spark
# spark = SparkSession.builder.appName("GroupMembershipETL").getOrCreate()
spark = SparkSession.builder.appName("BronzeToSilver_ListsLibrary").getOrCreate()

today = datetime.now()  
from datetime import datetime, timedelta
today = today - timedelta(days=1)
year = today.strftime("%Y")
month = today.strftime("%m")
day = today.strftime("%d") 

group_path = f"{source_path}/{year}/{month}/{day}/Group_Memberships.csv"
user_path = f"{source_path}/{year}/{month}/{day}/User_Group_Memberships.csv"

current_date = datetime.now()
year, month, day = current_date.strftime("%Y"), current_date.strftime("%m"), current_date.strftime("%d")
output_path = f"{target_path}/{year}/{month}/{day}/Dim_Group_Membership.parquet"

# Read CSVs
group_df = spark.read.option("header", True).csv(group_path)
user_df = spark.read.option("header", True).csv(user_path)

StatementMeta(, 6dc5032b-58cb-4cfe-a66c-cb1917b73da0, 5, Finished, Available, Finished)

In [4]:
group_df.printSchema()
user_df.printSchema()

StatementMeta(, 6dc5032b-58cb-4cfe-a66c-cb1917b73da0, 6, Finished, Available, Finished)

root
 |-- GroupName: string (nullable = true)
 |-- GroupId: string (nullable = true)
 |-- GroupDescription: string (nullable = true)
 |-- GroupOwner: string (nullable = true)
 |-- GroupAllowMembersEditMembership: string (nullable = true)
 |-- GroupOnlyAllowMembersViewMembership: string (nullable = true)
 |-- MemberName: string (nullable = true)
 |-- MemberEmail: string (nullable = true)
 |-- MemberLogin: string (nullable = true)
 |-- MemberId: string (nullable = true)
 |-- MemberType: string (nullable = true)
 |-- MemberUserPrincipalName: string (nullable = true)
 |-- IsExternal: string (nullable = true)
 |-- IsGuest: string (nullable = true)
 |-- IsSystemAccount: string (nullable = true)
 |-- GroupPermissionSummary: string (nullable = true)
 |-- GroupPermissionCount: string (nullable = true)
 |-- MemberAddedDate: string (nullable = true)
 |-- MemberAddedBy: string (nullable = true)
 |-- LastReviewedDate: string (nullable = true)
 |-- ReviewedBy: string (nullable = true)
 |-- CapturedD

In [5]:

# Add metadata
group_df = group_df.withColumn("SnapshotDate", current_timestamp()) \
                   .withColumn("ProcessedDate", current_timestamp()) \
                   .withColumn("DataSource", lit("SharePoint"))

user_df = user_df.withColumn("SnapshotDate", current_timestamp()) \
                 .withColumn("ProcessedDate", current_timestamp()) \
                 .withColumn("DataSource", lit("SharePoint"))

# Convert all columns to string
def cast_all_to_string(df):
    for field in df.schema.fields:
        if field.name not in ["SnapshotDate", "ProcessedDate", "DataSource"]:
            df = df.withColumn(field.name, col(field.name).cast(StringType()))
    return df

# Trim string columns
def trim_columns(df):
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            df = df.withColumn(field.name, trim(col(field.name)))
    return df

# Normalize boolean columns
def normalize_boolean(df, columns):
    for col_name in columns:
        if col_name in df.columns:
            df = df.withColumn(
                col_name,
                when(upper(col(col_name)).isin(["TRUE", "1", "YES", "Y"]), lit(True))
                .when(upper(col(col_name)).isin(["FALSE", "0", "NO", "N"]), lit(False))
                .otherwise(None)
            )
    return df

# Normalize date columns
def normalize_dates(df, columns):
    for col_name in columns:
        if col_name in df.columns:
            df = df.withColumn(col_name, to_timestamp(col(col_name), "M/d/yyyy H:mm"))
    return df

# Normalize numeric columns
def normalize_numeric(df, columns):
    for col_name in columns:
        if col_name in df.columns:
            df = df.withColumn(col_name, regexp_replace(col(col_name), "[^0-9]", "").cast(IntegerType()))
    return df

# Apply transformations
group_df = cast_all_to_string(group_df)
user_df = cast_all_to_string(user_df)

group_df = trim_columns(group_df)
user_df = trim_columns(user_df)

null_values = ["", "NULL", "null", "N/A", "n/a"]
for val in null_values:
    group_df = group_df.replace(val, None)
    user_df = user_df.replace(val, None)

group_df = normalize_boolean(group_df, ["IsExternal", "IsExternalUser"])
user_df = normalize_boolean(user_df, ["IsOwner", "IsActive"])

# Rename conflicting columns
group_df = group_df \
    .withColumnRenamed("SnapshotDate", "SnapshotDate_group") \
    .withColumnRenamed("ProcessedDate", "ProcessedDate_group") \
    .withColumnRenamed("DataSource", "DataSource_group") \
    .withColumnRenamed("CreatedDate", "CreatedDate_group") \
    .withColumnRenamed("ModifiedDate", "ModifiedDate_group") \
    .withColumnRenamed("GroupId", "GroupId_group")

user_df = user_df \
    .withColumnRenamed("GroupId", "GroupId_user") \
    .withColumnRenamed("SnapshotDate", "SnapshotDate_user") \
    .withColumnRenamed("ProcessedDate", "ProcessedDate_user") \
    .withColumnRenamed("DataSource", "DataSource_user")

# group_df = normalize_dates(group_df, ["CreatedDate_group", "ModifiedDate_group", "SnapshotDate_group"])
# user_df = normalize_dates(user_df, ["CreatedDate", "ModifiedDate", "SnapshotDate_user"])

# group_df = normalize_numeric(group_df, ["GroupId_group"])
# user_df = normalize_numeric(user_df, ["UserId", "GroupId_user"])



StatementMeta(, 6dc5032b-58cb-4cfe-a66c-cb1917b73da0, 7, Finished, Available, Finished)

In [6]:
# Creating Primary Keys *****

from pyspark.sql.functions import col, when, trim, sha2, coalesce, concat, lit

def create_user_key(df, email_col, display_name_col, identifier_col="user_identifier", 
                   hash_col="UserKey", salt=None):
        
    # Step 1: Create identifier column with email as priority, fallback to display name
    df_with_identifier = df.withColumn(
        identifier_col,
        when(
            # Check if email is not null and not empty/blank
            (col(email_col).isNotNull()) & 
            (trim(col(email_col)) != "") & 
            (col(email_col) != " "),
            trim(col(email_col))  # Use email if valid
        ).otherwise(
            # Fallback to display name
            when(
                (col(display_name_col).isNotNull()) & 
                (trim(col(display_name_col)) != "") & 
                (col(display_name_col) != " "),
                trim(col(display_name_col))
            ).otherwise(None)  # Set to null if both are blank
        )
    )
    
    # Step 2: Create SHA256 hash of the identifier column
    if salt:
        # With salt for better security
        df_final = df_with_identifier.withColumn(
            hash_col,
            when(
                col(identifier_col).isNotNull(),
                sha2(concat(lit(salt), col(identifier_col)), 256)
            ).otherwise(None)
        )
    else:
        # Without salt
        df_final = df_with_identifier.withColumn(
            hash_col,
            when(
                col(identifier_col).isNotNull(),
                sha2(col(identifier_col), 256)
            ).otherwise(None)
        )
    
    return df_final

# Custom column names
user_df = create_user_key(
    df=user_df,
    email_col="UserEmail",
    display_name_col="UserDisplayName",
    identifier_col="user_uniqueid",
    hash_col="UserKey"
)

user_df = create_user_key(
    df=user_df,
    email_col="GroupLoginName",
    display_name_col="GroupLoginName",
    identifier_col="group_uniqueid",
    hash_col="GroupKey"
)

group_df = create_user_key(
    df=group_df,
    email_col="MemberEmail",
    display_name_col="MemberName",
    identifier_col="user_uniqueid",
    hash_col="UserKey"
)

group_df = create_user_key(
    df=group_df,
    email_col="GroupName",
    display_name_col="GroupName",
    identifier_col="group_uniqueid",
    hash_col="GroupKey"
)

user_df.printSchema()
group_df.printSchema()

StatementMeta(, 6dc5032b-58cb-4cfe-a66c-cb1917b73da0, 8, Finished, Available, Finished)

root
 |-- MembershipId: string (nullable = true)
 |-- UserId: string (nullable = true)
 |-- UserLoginName: string (nullable = true)
 |-- UserDisplayName: string (nullable = true)
 |-- GroupId_user: string (nullable = true)
 |-- GroupLoginName: string (nullable = true)
 |-- GroupDisplayName: string (nullable = true)
 |-- UserEmail: string (nullable = true)
 |-- SiteId: string (nullable = true)
 |-- SiteName: string (nullable = true)
 |-- SiteUrl: string (nullable = true)
 |-- WebId: string (nullable = true)
 |-- MembershipType: string (nullable = true)
 |-- IsOwner: boolean (nullable = true)
 |-- IsMember: string (nullable = true)
 |-- GroupType: string (nullable = true)
 |-- MembershipSource: string (nullable = true)
 |-- JoinedDateTime: string (nullable = true)
 |-- AddedBy: string (nullable = true)
 |-- UserDepartment: string (nullable = true)
 |-- UserJobTitle: string (nullable = true)
 |-- UserLocation: string (nullable = true)
 |-- UserCountry: string (nullable = true)
 |-- Inheri

In [7]:
# Basic join with default settings
# Join with explicit column references using aliases

final_df = user_df.alias("u").join(
    group_df.alias("g"),
    (col("u.UserKey") == col("g.UserKey")) & (col("u.GroupKey") == col("g.GroupKey")),
    how="left"
).select(
    col("u.*"),
    col("g.GroupOwner"),
    col("g.MemberType"),
    col("g.IsExternal"),
    col("g.GroupPermissionSummary")
)

final_df.printSchema()


StatementMeta(, 6dc5032b-58cb-4cfe-a66c-cb1917b73da0, 9, Finished, Available, Finished)

root
 |-- MembershipId: string (nullable = true)
 |-- UserId: string (nullable = true)
 |-- UserLoginName: string (nullable = true)
 |-- UserDisplayName: string (nullable = true)
 |-- GroupId_user: string (nullable = true)
 |-- GroupLoginName: string (nullable = true)
 |-- GroupDisplayName: string (nullable = true)
 |-- UserEmail: string (nullable = true)
 |-- SiteId: string (nullable = true)
 |-- SiteName: string (nullable = true)
 |-- SiteUrl: string (nullable = true)
 |-- WebId: string (nullable = true)
 |-- MembershipType: string (nullable = true)
 |-- IsOwner: boolean (nullable = true)
 |-- IsMember: string (nullable = true)
 |-- GroupType: string (nullable = true)
 |-- MembershipSource: string (nullable = true)
 |-- JoinedDateTime: string (nullable = true)
 |-- AddedBy: string (nullable = true)
 |-- UserDepartment: string (nullable = true)
 |-- UserJobTitle: string (nullable = true)
 |-- UserLocation: string (nullable = true)
 |-- UserCountry: string (nullable = true)
 |-- Inheri

In [8]:

# silver_source = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting"
# temp_path = f"{silver_source}/temp/{year}/{month}/{day}/Dim_GroupMembership_temp.parquet"

# # Read temp parquet file
# df_temp = spark.read.parquet(temp_path)

# Drop intermediate columns with suffixes
# columns_to_drop = [col_name for col_name in df_temp.columns if col_name.endswith("_user") or col_name.endswith("_group")]
# df_cleaned = df_temp.drop(*columns_to_drop)
# df_cleaned = df_cleaned.withColumnRenamed("RecordedDateTime", "SnapshotDate")

final_df = final_df.withColumnRenamed("SnapshotDate_user", "SnapshotDate")
final_df = final_df.withColumnRenamed("ProcessedDate_user", "ProcessedDate")
final_df = final_df.withColumnRenamed("DataSource_user", "DataSource")

# final_df = final_df.withColumnRenamed("UserId", "UserID")
final_df = final_df.withColumnRenamed("GroupId_user", "GroupId")

# Save cleaned DataFrame to final path
output_path = f"{target_path}/{year}/{month}/{day}/Dim_Group_Membership.parquet"
final_df.write.mode("overwrite").option("compression", "snappy").parquet(output_path)

# Optional: Read back the saved parquet to verify
df_final = spark.read.parquet(output_path)

# Show sample
df_final.show(2, truncate=False)
final_df.printSchema()

StatementMeta(, 6dc5032b-58cb-4cfe-a66c-cb1917b73da0, 10, Finished, Available, Finished)

+------------------------------------+------+--------------------------------------------------+-----------------+-------+-------------------------------------------------------------------------+-------------------------------------------------------------------------+--------------------------------+------------------------------------+-------------+------------------------------------------------------------+------------------------------------+--------------+-------+--------+----------+----------------+--------------+-------+--------------+---------------------------------------------------+------------+-----------+--------------------------+--------+---------------------+------------+--------------------------+--------------------------+----------+--------------------------------+----------------------------------------------------------------+-------------------------------------------------------------------------+----------------------------------------------------------------+

In [9]:
from datetime import datetime

# Define the log_etl_activity function for logging ETL process
def log_etl_activity(status, start_time, rows_read=None, rows_written=None, bytes_processed=None, error_details=None):

    end_time = datetime.now()
    duration_seconds = (end_time - start_time).total_seconds()

    log_message = {
        'Status': status,
        'StartTime': start_time,
        'EndTime': end_time,
        'DurationSeconds': duration_seconds,
        'RowsRead': rows_read,
        'RowsWritten': rows_written,
        'BytesProcessed': bytes_processed,
        'ErrorDetails': error_details
    }

    # For simplicity, let's print the log message (this can be replaced with a logging system)
    print("Logging ETL Activity:", log_message)

# Ensure processing_successful is defined before this block
try:
    NOTEBOOK_NAME = "ETL_Pipeline_Example"  # Define your notebook name or use the existing one
    start_time = datetime.now()  # Capture the start time of the ETL process

    print(f"🔄 Starting ETL processing for {NOTEBOOK_NAME}...")

    # Simulated metrics
    rows_read = group_df.count()   # Correct this to have a meaningful `rows_read`
    rows_read = user_df.count()
    rows_written = final_df.count()
    # file_count = len(dbutils.fs.ls(SOURCE_PATH))
    bytes_processed = 524288000  # ~500MB

    processing_successful = True

except Exception as e:
    error_details = e
    processing_successful = False

# Complete the logging based on processing results
if processing_successful:
    # Log successful completion with metrics
    log_etl_activity("SUCCESS", start_time, 
                     rows_read=rows_read, 
                     rows_written=rows_written,
                     bytes_processed=bytes_processed)
    
    print(f"🎉 {NOTEBOOK_NAME} pipeline completed successfully!")
    print(f"📊 Final metrics:")
    print(f"   ✅ Status: SUCCESS")
    print(f"   📖 Total rows processed: {rows_read:,} → {rows_written:,}")
    print(f"   🔄 Data throughput: {bytes_processed/(1024**2):.1f} MB")
    
    # Optional: Show recent logs for this notebook
    print(f"\n📋 Recent runs for {NOTEBOOK_NAME}:")
    spark.sql(f"""
        SELECT LogID, Status, StartTime, EndTime, DurationSeconds, RowsRead, RowsWritten
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=True)
    
else:
    # Log failure
    log_etl_activity("FAILED", start_time, error_details=error_details)
    
    print(f"💥 {NOTEBOOK_NAME} pipeline failed!")
    print(f"❌ Error: {str(error_details)}")
    
    # Optional: Show error analysis
    print(f"\n🔍 Recent failures for debugging:")
    spark.sql(f"""
        SELECT LogID, StartTime, ErrorMessage, DurationSeconds
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}' AND Status = 'FAILED'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=False)
    
    # Re-raise the exception to fail the notebook
    raise error_details

# Cleanup variables
print(f"\n🧹 Cleaning up variables...")
del rows_read, rows_written, bytes_processed

print(f"✨ {NOTEBOOK_NAME} logging completed!")


StatementMeta(, 6dc5032b-58cb-4cfe-a66c-cb1917b73da0, 11, Finished, Available, Finished)

🔄 Starting ETL processing for ETL_Pipeline_Example...
Logging ETL Activity: {'Status': 'SUCCESS', 'StartTime': datetime.datetime(2025, 10, 15, 5, 20, 52, 484152), 'EndTime': datetime.datetime(2025, 10, 15, 5, 20, 57, 2419), 'DurationSeconds': 4.518267, 'RowsRead': 333, 'RowsWritten': 333, 'BytesProcessed': 524288000, 'ErrorDetails': None}
🎉 ETL_Pipeline_Example pipeline completed successfully!
📊 Final metrics:
   ✅ Status: SUCCESS
   📖 Total rows processed: 333 → 333
   🔄 Data throughput: 500.0 MB

📋 Recent runs for ETL_Pipeline_Example:
+-----+------+---------+-------+---------------+--------+-----------+
|LogID|Status|StartTime|EndTime|DurationSeconds|RowsRead|RowsWritten|
+-----+------+---------+-------+---------------+--------+-----------+
+-----+------+---------+-------+---------------+--------+-----------+


🧹 Cleaning up variables...
✨ ETL_Pipeline_Example logging completed!
